# Phase 1 Theory: AWS Security APIs & Data Collection Patterns

**Time Estimate:** 4-5 hours | **Prerequisites:** Phase 0 completed

---

## Learning Objectives
By the end of this notebook you will understand:
1. How AWS Security Hub Finding Format (ASFF) works — the universal data model for security findings
2. How AWS Config records resource configurations and evaluates compliance rules
3. How to design an event-driven data collection pipeline using Lambda + EventBridge
4. Boto3 patterns for efficient, paginated API calls at scale
5. Error handling and retry strategies for AWS SDK calls

### Study Cross-References
| Concept | DDIA Chapter | DVA-C02 Domain | System Design Interview |
|---------|-------------|----------------|------------------------|
| Data serialization (ASFF is JSON) | Ch. 4: Encoding and Evolution | Domain 1: AWS SDKs | Ch. 7: Unique ID Generator (ID design) |
| Event-driven architecture | Ch. 11: Stream Processing | Domain 1: Lambda, EventBridge | Ch. 11: News Feed (pub/sub) |
| Idempotency | Ch. 11: Exactly-once semantics | Domain 1: Lambda idempotency | Ch. 5: Consistent Hashing |
| Pagination & batching | Ch. 3: SSTables & LSM-Trees | Domain 1: SDK pagination | — |

---

## Section 1: AWS Security Hub Finding Format (ASFF)

### 1.1 Why ASFF Matters

The AWS Security Finding Format (ASFF) is the **canonical data model** for all security findings in AWS. Every finding from Security Hub, GuardDuty, Inspector, Macie, and third-party integrations gets normalized into ASFF.

**DDIA Connection (Ch. 4 — Encoding and Evolution):** ASFF is a schema evolution success story. AWS needed a single format that could represent findings from dozens of services, each with different data shapes. They solved it with a semi-structured JSON schema that has required fields plus extensible `Resources` and `ProductFields` objects. This is the same forward/backward compatibility challenge Kleppmann discusses.

### 1.2 ASFF Structure — The Fields That Matter

An ASFF finding has ~100 possible fields, but for compliance evidence, these are the ones you care about:

```json
{
  "SchemaVersion": "2018-10-08",
  "Id": "arn:aws:securityhub:us-east-1:123456789:subscription/cis-aws-foundations/v/1.2.0/1.1/finding/abc-123",
  "ProductArn": "arn:aws:securityhub:us-east-1::product/aws/securityhub",
  "GeneratorId": "cis-aws-foundations-benchmark/v/1.2.0/1.1",
  
  "AwsAccountId": "123456789012",
  "Region": "us-east-1",
  
  "Types": ["Software and Configuration Checks/Industry and Regulatory Standards"],
  
  "Severity": {
    "Label": "HIGH",
    "Normalized": 70
  },
  
  "Title": "1.1 Avoid the use of the root account",
  "Description": "The root account has unrestricted access to all resources...",
  
  "Compliance": {
    "Status": "FAILED",
    "RelatedRequirements": [
      "NIST.800-53.r5 AC-2",
      "NIST.800-53.r5 AC-6(5)",
      "CIS AWS Foundations Benchmark v1.4.0/1.7"
    ],
    "SecurityControlId": "IAM.4",
    "AssociatedStandards": [
      {"StandardsId": "standards/cis-aws-foundations-benchmark/v/1.4.0"}
    ]
  },
  
  "Resources": [
    {
      "Type": "AwsAccount",
      "Id": "AWS::::Account:123456789012",
      "Partition": "aws",
      "Region": "us-east-1"
    }
  ],
  
  "RecordState": "ACTIVE",
  "WorkflowState": "NEW",
  
  "CreatedAt": "2024-01-15T10:30:00.000Z",
  "UpdatedAt": "2024-01-15T10:30:00.000Z",
  
  "Remediation": {
    "Recommendation": {
      "Text": "Enable MFA on the root account and avoid using it for daily operations.",
      "Url": "https://docs.aws.amazon.com/..."
    }
  }
}
```

### 1.3 The Critical Field: `Compliance.RelatedRequirements`

This is the **golden field** for your tool. AWS has already done the work of mapping many Security Hub findings to NIST 800-53 control IDs. The `RelatedRequirements` array contains strings like `"NIST.800-53.r5 AC-2"` that directly map findings to controls.

**However, the mapping is incomplete.** AWS covers maybe 60-70% of automatable controls. Your tool's value-add is:
1. Filling in the remaining 30-40% with custom mappings
2. Pulling evidence from services that don't feed into Security Hub
3. Formatting everything into auditor-friendly PDFs

### Documentation
- [ASFF Complete Reference](https://docs.aws.amazon.com/securityhub/latest/userguide/securityhub-findings-format.html)
- [ASFF Compliance Object](https://docs.aws.amazon.com/securityhub/latest/userguide/asff-compliance.html)
- [Security Hub Standards Reference](https://docs.aws.amazon.com/securityhub/latest/userguide/standards-reference.html)

## Section 2: AWS Config — The Configuration Time Machine

### 2.1 How AWS Config Works Under the Hood

**DDIA Connection (Ch. 3 — Storage and Retrieval):** AWS Config is conceptually an **append-only log** (like a write-ahead log) combined with a **materialized view** (the current configuration snapshot).

```
Event Timeline:
T1: S3 bucket created (encryption=None)
T2: Config records ConfigurationItem {encryption: null}
T3: Config rule evaluates → NON_COMPLIANT
T4: Someone enables encryption on the bucket
T5: Config records new ConfigurationItem {encryption: AES256}
T6: Config rule re-evaluates → COMPLIANT

You can query ANY point in this timeline:
- "What was the config at T2?" → {encryption: null}
- "What was the config at T5?" → {encryption: AES256}
- "When did it become compliant?" → T6
```

This is exactly the **event sourcing** pattern. The full history of changes is preserved, and you can derive the state at any point. Kleppmann covers this in DDIA Ch. 11 (Stream Processing → Event Sourcing).

### 2.2 Config Rules — The Compliance Evaluators

Config Rules evaluate whether resources comply with your desired configurations. There are three types:

| Type | How It Works | Example | Your Use |
|------|-------------|---------|----------|
| **AWS Managed Rules** | Pre-built by AWS, just enable | `s3-bucket-server-side-encryption-enabled` | Primary source — use ~50 managed rules |
| **Custom Lambda Rules** | Your Lambda evaluates compliance | Check if EC2 has specific tags | For controls AWS doesn't cover |
| **CloudFormation Guard Rules** | Policy-as-code evaluation | Validate JSON configs against rules | Advanced use cases |

**Key managed rules for NIST 800-53:**

```
Access Control (AC):
  - iam-root-access-key-check
  - iam-user-mfa-enabled
  - iam-user-no-policies-check
  - iam-policy-no-statements-with-admin-access

Audit & Accountability (AU):
  - cloud-trail-cloud-watch-logs-enabled
  - cloudtrail-enabled
  - multi-region-cloudtrail-enabled
  - cloud-trail-log-file-validation-enabled

Configuration Management (CM):
  - ec2-instance-managed-by-systems-manager
  - ec2-managedinstance-patch-compliance-status-check
  - restricted-ssh

System & Comms Protection (SC):
  - s3-bucket-server-side-encryption-enabled
  - s3-bucket-ssl-requests-only
  - encrypted-volumes
  - rds-storage-encrypted
  - elasticsearch-encrypted-at-rest

System & Info Integrity (SI):
  - guardduty-enabled-centralized
  - securityhub-enabled
```

### Documentation
- [AWS Config Managed Rules List (full)](https://docs.aws.amazon.com/config/latest/developerguide/managed-rules-by-aws-config.html)
- [AWS Config Developer Guide](https://docs.aws.amazon.com/config/latest/developerguide/WhatIsConfig.html)
- [Config Advanced Queries (SQL syntax)](https://docs.aws.amazon.com/config/latest/developerguide/querying-AWS-resources.html)

## Section 3: Event-Driven Collection Architecture

### 3.1 Why Event-Driven?

**DVA-C02 Exam Alert:** Event-driven architecture with Lambda + EventBridge is a major exam topic.

Two approaches to collecting compliance data:

| Approach | Pros | Cons |
|----------|------|------|
| **Scheduled polling** (EventBridge cron → Lambda) | Simple, predictable, easy to debug | Misses real-time changes, higher latency |
| **Event-driven** (Config changes → EventBridge → Lambda) | Real-time, efficient | More complex, needs dead-letter queues |

**Our approach: Both.** Use scheduled scans for comprehensive baseline snapshots (every 6 hours) AND event-driven triggers for critical changes (IAM policy modifications, security group changes).

### 3.2 The Collection Lambda Design

```
EventBridge Rule (cron: 0 */6 * * *)  ─────┐
                                              │
EventBridge Rule (Config change event) ──────┤
                                              │
Security Hub finding imported event ─────────┤
                                              ▼
                                    ┌─────────────────┐
                                    │ collector_lambda │
                                    │                 │
                                    │ 1. Generate     │
                                    │    scan_id      │
                                    │ 2. Pull from    │
                                    │    each service │
                                    │ 3. Normalize    │
                                    │    data         │
                                    │ 4. Store raw    │
                                    │    evidence     │
                                    │ 5. Trigger      │
                                    │    mapper       │
                                    └─────────────────┘
                                              │
                            ┌─────────────────┼──────────────┐
                            ▼                 ▼              ▼
                    ┌──────────────┐  ┌──────────────┐  ┌──────────┐
                    │ DynamoDB     │  │ S3 raw/      │  │ mapping_ │
                    │ findings     │  │ evidence     │  │ lambda   │
                    └──────────────┘  └──────────────┘  └──────────┘
```

### 3.3 Lambda Best Practices for This Project

**DVA-C02 must-know topics:**

**Cold Starts:** Our collector Lambda will be invoked every 6 hours (not frequently enough to stay warm). Mitigation strategies:
- Use Python 3.12 (faster cold start than Java/C#)
- Keep deployment package small (only import what you need)
- Initialize boto3 clients OUTSIDE the handler function (reused across warm invocations)
- Consider Provisioned Concurrency if latency matters (it doesn't much for us — compliance isn't real-time)

```python
# GOOD — client initialized outside handler (reused when warm)
import boto3
securityhub_client = boto3.client('securityhub')

def handler(event, context):
    findings = securityhub_client.get_findings()
    # ...

# BAD — client created on every invocation
def handler(event, context):
    client = boto3.client('securityhub')  # Wasted time on warm starts
    findings = client.get_findings()
```

**Timeout Configuration:** Our collector pulls from multiple services and might take 2-5 minutes. Set Lambda timeout to 15 minutes (maximum). For the DVA-C02: know that the max Lambda timeout is 15 minutes.

**Memory:** More memory = more CPU = faster execution. For our collector doing network I/O:
- 256MB is sufficient for light collection
- 512MB-1024MB for environments with thousands of findings
- DVA-C02 tip: Lambda allocates CPU proportional to memory. At 1,769MB you get 1 full vCPU.

**Idempotency:**

**DDIA Connection (Ch. 11 — Exactly-once semantics):** Lambda can be invoked more than once (at-least-once delivery). Our collector must be idempotent — running it twice with the same scan_id should produce the same result without duplicating data.

Strategy: Use the `scan_id` (containing timestamp) as a unique key. Before writing, check if data for that scan_id already exists. Use DynamoDB conditional writes.

```python
# Idempotent write using DynamoDB conditional expression
table.put_item(
    Item={'PK': f'SCAN#{scan_id}', 'SK': f'FINDING#{finding_id}', ...},
    ConditionExpression='attribute_not_exists(PK)'  # Only write if not exists
)
```

### Documentation
- [Lambda Developer Guide](https://docs.aws.amazon.com/lambda/latest/dg/welcome.html)
- [Lambda Best Practices](https://docs.aws.amazon.com/lambda/latest/dg/best-practices.html)
- [EventBridge User Guide](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-what-is.html)
- [Lambda Powertools for Python (idempotency)](https://docs.powertools.aws.dev/lambda/python/latest/utilities/idempotency/)

## Section 4: Boto3 Patterns You Must Master

### 4.1 Pagination — The #1 Boto3 Gotcha

**DVA-C02 Exam Alert:** You WILL be tested on pagination. Many AWS APIs return paginated results. If you don't paginate, you miss data.

```python
# WRONG — only gets first page (max 100 findings)
response = securityhub.get_findings()
findings = response['Findings']  # Missing potentially thousands of findings!

# RIGHT — using paginator
paginator = securityhub.get_paginator('get_findings')
all_findings = []
for page in paginator.paginate(
    Filters={'ComplianceStatus': [{'Value': 'FAILED', 'Comparison': 'EQUALS'}]}
):
    all_findings.extend(page['Findings'])

# RIGHT — manual pagination (when paginator not available)
findings = []
next_token = None
while True:
    kwargs = {'MaxResults': 100}
    if next_token:
        kwargs['NextToken'] = next_token
    response = securityhub.get_findings(**kwargs)
    findings.extend(response['Findings'])
    next_token = response.get('NextToken')
    if not next_token:
        break
```

### 4.2 Error Handling and Retries

AWS APIs can fail. Your collector must handle this gracefully.

```python
from botocore.exceptions import ClientError
from botocore.config import Config
import time

# Configure automatic retries with exponential backoff
config = Config(
    retries={
        'max_attempts': 5,
        'mode': 'adaptive'  # DVA-C02: know standard vs adaptive retry modes
    }
)
client = boto3.client('securityhub', config=config)

# Manual error handling for specific cases
try:
    response = client.get_findings(
        Filters={'ComplianceStatus': [{'Value': 'FAILED', 'Comparison': 'EQUALS'}]}
    )
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == 'ThrottlingException':
        # Exponential backoff — DDIA Ch. 8 discusses this pattern
        time.sleep(2 ** attempt_number)
        # retry...
    elif error_code == 'AccessDeniedException':
        # IAM permission issue — log and skip this service
        logger.error(f"Missing permissions for Security Hub: {e}")
    elif error_code == 'InvalidAccessException':
        # Security Hub not enabled in this region
        logger.warning("Security Hub not enabled")
    else:
        raise  # Unknown error — let it bubble up
```

### 4.3 Filtering Findings Efficiently

Security Hub can have thousands of findings. Always filter server-side, not client-side:

```python
# GOOD — server-side filtering (less data transferred, faster)
paginator = securityhub.get_paginator('get_findings')
for page in paginator.paginate(
    Filters={
        'ComplianceStatus': [
            {'Value': 'FAILED', 'Comparison': 'EQUALS'}
        ],
        'RecordState': [
            {'Value': 'ACTIVE', 'Comparison': 'EQUALS'}
        ],
        'WorkflowStatus': [
            {'Value': 'NEW', 'Comparison': 'EQUALS'},
            {'Value': 'NOTIFIED', 'Comparison': 'EQUALS'}
        ],
        'SeverityLabel': [
            {'Value': 'HIGH', 'Comparison': 'EQUALS'},
            {'Value': 'CRITICAL', 'Comparison': 'EQUALS'}
        ]
    },
    SortCriteria=[{'Field': 'SeverityNormalized', 'SortOrder': 'desc'}]
):
    process_findings(page['Findings'])

# BAD — client-side filtering (pulls everything, then filters)
all_findings = get_all_findings()  # Could be 10,000 items
failed = [f for f in all_findings if f['Compliance']['Status'] == 'FAILED']
```

### Documentation
- [Boto3 Paginators](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/paginators.html)
- [Boto3 Retries](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/retries.html)
- [AWS SDK Exponential Backoff](https://docs.aws.amazon.com/general/latest/gr/api-retries.html)

## Section 5: IAM for the Collector — Least Privilege in Practice

### 5.1 The Collector's IAM Role

**DVA-C02 Exam Alert:** IAM policy design is heavily tested. Know the difference between identity-based and resource-based policies.

Your collector Lambda needs a carefully scoped IAM role. Here's the principle of least privilege applied:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "SecurityHubReadOnly",
      "Effect": "Allow",
      "Action": [
        "securityhub:GetFindings",
        "securityhub:GetEnabledStandards",
        "securityhub:DescribeStandardsControls",
        "securityhub:ListEnabledProductsForImport"
      ],
      "Resource": "*"
    },
    {
      "Sid": "ConfigReadOnly",
      "Effect": "Allow",
      "Action": [
        "config:GetComplianceDetailsByConfigRule",
        "config:DescribeComplianceByConfigRule",
        "config:GetResourceConfigHistory",
        "config:SelectResourceConfig",
        "config:DescribeConfigRules"
      ],
      "Resource": "*"
    },
    {
      "Sid": "IAMReadOnly",
      "Effect": "Allow",
      "Action": [
        "iam:GenerateCredentialReport",
        "iam:GetCredentialReport",
        "iam:ListUsers",
        "iam:GetAccountAuthorizationDetails",
        "iam:GetAccountPasswordPolicy",
        "iam:ListAttachedUserPolicies",
        "iam:GetAccountSummary"
      ],
      "Resource": "*"
    },
    {
      "Sid": "CloudTrailReadOnly",
      "Effect": "Allow",
      "Action": [
        "cloudtrail:DescribeTrails",
        "cloudtrail:GetTrailStatus",
        "cloudtrail:LookupEvents"
      ],
      "Resource": "*"
    },
    {
      "Sid": "GuardDutyReadOnly",
      "Effect": "Allow",
      "Action": [
        "guardduty:ListDetectors",
        "guardduty:GetDetector",
        "guardduty:ListFindings",
        "guardduty:GetFindings"
      ],
      "Resource": "*"
    },
    {
      "Sid": "WriteToStorage",
      "Effect": "Allow",
      "Action": [
        "s3:PutObject",
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::compliance-evidence-*/*"
    },
    {
      "Sid": "WriteToDynamoDB",
      "Effect": "Allow",
      "Action": [
        "dynamodb:PutItem",
        "dynamodb:GetItem",
        "dynamodb:Query",
        "dynamodb:BatchWriteItem"
      ],
      "Resource": "arn:aws:dynamodb:*:*:table/ComplianceData*"
    },
    {
      "Sid": "CloudWatchLogs",
      "Effect": "Allow",
      "Action": [
        "logs:CreateLogGroup",
        "logs:CreateLogStream",
        "logs:PutLogEvents"
      ],
      "Resource": "arn:aws:logs:*:*:log-group:/aws/lambda/compliance-collector*"
    }
  ]
}
```

**Notice:** Every statement has the minimum actions needed. Resource ARNs are scoped where possible (S3 bucket prefix, DynamoDB table prefix, specific log group). This is what auditors look for.

### 5.2 Trust Policy

The Lambda service needs to assume this role:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "lambda.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }
  ]
}
```

### Documentation
- [IAM Policy Reference](https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies.html)
- [Lambda Execution Role](https://docs.aws.amazon.com/lambda/latest/dg/lambda-intro-execution-role.html)
- [IAM Policy Simulator (test your policies)](https://policysim.aws.amazon.com/)

## Section 6: Testing Without AWS — The moto Library

You don't need a live AWS account to develop and test your collector. The `moto` library mocks AWS services locally.

**DVA-C02 relevance:** Understanding how to test AWS integrations is part of Domain 1.

```python
# Example: Testing Security Hub collection with moto
import boto3
from moto import mock_securityhub
import pytest

@mock_securityhub
def test_collect_security_hub_findings():
    # Setup: Create mock Security Hub with findings
    client = boto3.client('securityhub', region_name='us-east-1')
    client.enable_security_hub()
    
    # Import a mock finding
    client.batch_import_findings(
        Findings=[{
            'SchemaVersion': '2018-10-08',
            'Id': 'test-finding-001',
            'ProductArn': 'arn:aws:securityhub:us-east-1:123456789:product/aws/securityhub',
            'GeneratorId': 'test-generator',
            'AwsAccountId': '123456789012',
            'Types': ['Software and Configuration Checks'],
            'CreatedAt': '2024-01-15T10:00:00Z',
            'UpdatedAt': '2024-01-15T10:00:00Z',
            'Severity': {'Label': 'HIGH', 'Normalized': 70},
            'Title': 'Root account has no MFA',
            'Description': 'The root account does not have MFA enabled',
            'Resources': [{'Type': 'AwsAccount', 'Id': '123456789012'}],
            'Compliance': {
                'Status': 'FAILED',
                'RelatedRequirements': ['NIST.800-53.r5 IA-2(1)']
            }
        }]
    )
    
    # Act: Run our collector function
    from src.collector.security_hub import collect_findings
    results = collect_findings(client)
    
    # Assert
    assert len(results) == 1
    assert results[0]['Compliance']['Status'] == 'FAILED'
    assert 'NIST.800-53.r5 IA-2(1)' in results[0]['Compliance']['RelatedRequirements']
```

### Documentation
- [moto library — AWS mocking](https://github.com/getmoto/moto)
- [moto Security Hub docs](https://docs.getmoto.org/en/latest/docs/services/securityhub.html)
- [pytest with AWS](https://docs.aws.amazon.com/prescriptive-guidance/latest/best-practices-cdk-typescript-iac/development-best-practices.html)

## Section 7: Exercises & Checkpoints

### Conceptual

**1. What is ASFF and why did AWS create a universal finding format? How does this relate to schema evolution in DDIA Ch. 4?**

ASFF (Amazon Security Finding Format) is a standardized JSON schema that every AWS security service — GuardDuty, Inspector, Config, Macie, and third-party tools — uses to describe a finding. Before ASFF, each service had its own JSON structure, so aggregating findings meant writing a custom parser per service.

DDIA Ch. 4 connection: this is the schema evolution problem. When you have multiple producers writing to the same consumer, you need forward and backward compatibility — new fields added by one service shouldn't break readers expecting the old shape. ASFF solves this with a fixed envelope plus optional extension fields (`Resources`, `ProductFields`), the same pattern Kleppmann recommends. Your collector code can safely ignore unknown fields and still parse the mandatory ones.

---

**2. Explain the difference between a Config recording and a Config rule evaluation.**

- **Recording** is the continuous process of taking snapshots of your resource configurations and writing them to S3. It answers "what does this resource look like right now?"
- **Rule evaluation** is a separate process that compares a recorded configuration against a rule's logic and produces a PASS/FAIL verdict. It answers "does this resource comply with this standard?"

Recording must happen before evaluation — you can't evaluate a config you haven't recorded. They also run on different triggers: recording happens continuously or on change, while rule evaluation runs on schedule or on configuration change depending on how the rule is set up.

---

**3. Why do we use both scheduled and event-driven collection? What would we miss with only one approach?**

- **Scheduled alone** means you miss changes that happen between runs. A security group opened at 2am stays open until your 6am scan catches it — 4 hours of undetected drift.
- **Event-driven alone** means you only capture changes, never baseline state. If a resource was misconfigured before you set up the tool, you'd never see it because no change event fires for it.

Together: scheduled scans establish the full baseline; event-driven collection catches real-time drift between scans. For FedRAMP, auditors want continuous monitoring, not just periodic snapshots.

---

**4. What happens if our Lambda gets throttled mid-collection? How should we handle partial data?**

If Lambda gets throttled mid-run, you get a partial dataset with no record of where it stopped. Two strategies:

- **Checkpoint pattern**: write progress to DynamoDB after each paginated batch (record the `NextToken`). On retry, resume from the last checkpoint instead of restarting from scratch.
- **Idempotent writes**: structure DynamoDB writes so re-running the same collection overwrites rather than duplicates. Use the finding ID as the sort key so a retry is safe.

For very large environments, move to Step Functions — it handles state, retries, and partial failure natively across multiple Lambda invocations.

---

### DVA-C02 Practice

**1. Lambda timing out at 3 min / 128MB — what's the FIRST thing to try?**

Increase memory. Lambda allocates CPU proportionally to memory — at 128MB you get a fraction of a vCPU. Bumping to 512MB or 1GB often cuts execution time by 60-70%, more than enough to fix a timeout. Only after that should you look at code optimization or splitting the function. At 1,769MB you get exactly 1 full vCPU.

---

**2. Processing 50,000 findings with max 100 per call — correct approach?**

Use a **paginator**. boto3 paginators handle the `NextToken` loop automatically:

```python
paginator = securityhub.get_paginator('get_findings')
for page in paginator.paginate(Filters={...}):
    process(page['Findings'])
```

For 50,000 findings that's 500 API calls — fine for Lambda with a high enough timeout. If the environment is larger or latency matters, fan out with SQS: write each page's findings to a queue and process in parallel with multiple Lambda workers. For orchestrating the whole pipeline, Step Functions with a Map state handles fan-out cleanly.

---

**3. `rate(6 hours)` vs `cron(0 */6 * * ? *)` — which is correct?**

Both work. `rate(6 hours)` is simpler and starts 6 hours after the rule is created — use this for most cases. `cron(0 */6 * * ? *)` fires at fixed times (midnight, 6am, noon, 6pm UTC) regardless of when the rule was created — use this when scan timing matters, e.g., aligning evidence collection to an audit window. Note AWS cron expressions require `?` in either day-of-month or day-of-week (not both) — a common DVA-C02 gotcha.

---

### Hands-On (Do These Before Moving to the Lab Notebook)

**1. Security Hub → Findings — ASFF fields to identify:**

Open a finding and look for: `Title`, `Description`, `Severity.Label`, `Resources[0].Type`, `Resources[0].Id` (the ARN), `Compliance.Status`, `Compliance.RelatedRequirements` (NIST control IDs), `ProductArn`, `CreatedAt`, `UpdatedAt`. These are the exact fields your collector will extract.

**2. Config → Rules — what to note:**

Each rule shows its trigger type (periodic vs. change-triggered), which resource types it evaluates, and compliance breakdown. Noncompliant rules are your tool's primary targets. Note how rule names map to NIST families — IAM rules → AC/IA, S3 rules → SC, CloudTrail rules → AU.

**3. Config Advanced Query:**

```sql
SELECT resourceId, resourceType WHERE compliance.complianceType = 'NON_COMPLIANT'
```

If this returns nothing despite noncompliant rules showing in the Rules page, the compliance index is still syncing — it can lag 20-30 minutes behind rule evaluations. Run `SELECT resourceId, resourceType` with no filter first to confirm the query engine is working. This query is what your boto3 collector replicates via `config.select_resource_config()`.

**Real-world note (learned during setup):** The Advanced Query index and rule evaluation results are separate — rules can show Noncompliant while the query returns nothing. This is normal during the first hour after enabling Config. The compliance data your tool actually collects via `get_compliance_details_by_config_rule()` is available immediately once rules evaluate.

---

## Next: Phase 1 Lab Notebook

In the lab notebook, you'll write the actual collector code — pulling from each service, normalizing data, and storing it.

**File:** `phase-1-data-collection/01-lab-building-the-collector.ipynb`